<a href="https://colab.research.google.com/github/ha0-922/VLA/blob/main/SmolVLA_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 SmolVLA Inference 데모

## 모델 소개: SmolVLA
- **SmolVLA** (Small Vision-Language-Action Model)는 HuggingFace가 2025년 공개한 소형 VLA 모델이에요
- VLA = **Vision(이미지) + Language(언어 명령) + Action(로봇 동작)** 을 하나의 모델로 처리
- 파라미터 수: **450M** (GPT 계열 대비 매우 작음 → 코랩 T4에서도 실행 가능)
- 내부 구조: SmolVLM2 (비전-언어 백본) + Action Expert (행동 예측 헤드)
- 학습 데이터: HuggingFace LeRobot 플랫폼의 커뮤니티 로봇 데이터셋

## 전체 흐름
```
카메라 이미지 ─┐
               ├──▶ SmolVLA ──▶ 로봇 관절 action (6차원 벡터)
언어 명령    ─┘
```


In [ ]:
# LeRobot: HuggingFace의 로보틱스 라이브러리 (모델, 데이터셋, 정책 포함)
# num2words: SmolVLA 내부 언어 처리기에 필요한 의존성
!pip install lerobot num2words -q

## Step 1. SmolVLA 모델 로드
- 출처: HuggingFace Hub `lerobot/smolvla_base`
- SmolVLM2-500M-Video-Instruct (비전-언어 백본) + Action Expert 두 파트로 구성
- 총 모델 크기: 약 3GB (백본 2GB + 액션 헤드 1GB)

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("디바이스:", device)

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
policy = policy.to(device)
policy.eval()
print("Loaded SmolVLA")

## Step 2. 데이터셋 로드
- 출처: HuggingFace Hub `lerobot/aloha_sim_insertion_scripted`
- ALOHA 로봇 시뮬레이터에서 수집한 데이터셋
- 태스크: **"Insert the peg into the socket"** (페그를 소켓에 꽂기)
- 각 샘플 구성:
  - `observation.images.top`: 위쪽 카메라 이미지 (3 × H × W)
  - `observation.state`: 로봇 현재 관절 상태 (6차원)
  - `action`: 정답 행동 레이블 (6차원)
  - `task`: 언어 명령 문자열

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset("lerobot/aloha_sim_insertion_scripted")
print("Finished loading data")
print("Dataset size:", len(dataset))

## Step 3. 추론 (Inference)
### 처리 단계:
1. **이미지 전처리**: 카메라 이미지를 텐서로 변환 → 모델 입력 형식에 맞게 정규화
2. **언어 토크나이징**: "Insert the peg into the socket" 문장을 토큰 ID 시퀀스로 변환
3. **SmolVLA 추론**:
   - 비전 백본이 이미지에서 시각적 특징 추출
   - 언어 인코더가 명령을 임베딩으로 변환
   - Action Expert가 두 정보를 융합해 다음 행동 예측
4. **출력**: 로봇 관절 6개의 이동값 (action vector)


In [ ]:
from transformers import AutoProcessor

# SmolVLA가 내부적으로 사용하는 언어 프로세서 로드
processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Video-Instruct")

sample = dataset[0]
task = sample["task"]
print("언어 명령:", task)

# 언어 명령 토크나이징 (텍스트 → 숫자 시퀀스)
tokens = processor.tokenizer(
    task,
    return_tensors="pt",
    padding="max_length",
    max_length=64,
    truncation=True
)

# 이미지 & 상태 준비
image_input = sample["observation.images.top"].unsqueeze(0).to(device)
state_input = sample["observation.state"].unsqueeze(0).to(device)

# SmolVLA는 camera1/2/3 키 이름을 기대하므로 동일 이미지로 채움(1 frame 확인용)
batch = {
    "observation.images.camera1": image_input,
    "observation.images.camera2": image_input,
    "observation.images.camera3": image_input,
    "observation.state": state_input,
    "task": [task],
    "observation.language.tokens": tokens["input_ids"].to(device),
    "observation.language.attention_mask": tokens["attention_mask"].bool().to(device),
}

# 추론
with torch.no_grad():
    action = policy.select_action(batch)

print("Inference complete")
print("언어 명령:", task)
print("action shape:", action.shape)  # (1, 6) = 배치 1개, 관절 6개
print("action:", action)

## Step 4. 연속 추론 & 시각화
- 20개 프레임에 걸쳐 연속으로 추론
- 각 프레임마다 SmolVLA가 이미지를 보고 언어 명령을 참고해서 action을 예측
- 관절 6개의 값이 시간에 따라 어떻게 변하는지 그래프로 확인
- **부드럽게 변하면** 모델이 일관성 있는 동작을 계획하고 있다는 의미

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

actions = []
n_frames = 20

policy.reset()

for i in range(n_frames):
    sample = dataset[i]
    task = sample["task"]
    image_input = sample["observation.images.top"].unsqueeze(0).to(device)
    state_input = sample["observation.state"].unsqueeze(0).to(device)

    tokens = processor.tokenizer(
        task,
        return_tensors="pt",
        padding="max_length",
        max_length=64,
        truncation=True
    )

    batch = {
        "observation.images.camera1": image_input,
        "observation.images.camera2": image_input,
        "observation.images.camera3": image_input,
        "observation.state": state_input,
        "task": [task],
        "observation.language.tokens": tokens["input_ids"].to(device),
        "observation.language.attention_mask": tokens["attention_mask"].bool().to(device),
    }

    with torch.no_grad():
        action = policy.select_action(batch)

    actions.append(action.squeeze(0).cpu().numpy())

actions = np.array(actions)  # (20, 6)

# 시각화
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
fig.suptitle(
    f"SmolVLA (lerobot/smolvla_base) Inference Result\n"
    f"Dataset: lerobot/aloha_sim_insertion_scripted\n"
    f"Task: '{task}'",
    fontsize=11
)

joint_names = ["Joint 1 (X)", "Joint 2 (Y)", "Joint 3 (Z)",
               "Joint 4 (Roll)", "Joint 5 (Pitch)", "Joint 6 (Yaw)"]

for i, ax in enumerate(axes.flat):
    ax.plot(actions[:, i], marker='o', color=f"C{i}", linewidth=2)
    ax.set_title(joint_names[i])
    ax.set_xlabel("Frame (time →)")
    ax.set_ylabel("Action Value")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("smolvla_actions.png", dpi=100)
plt.show()
print("✅ 'smolvla_actions.png' successfully generated.")

## Step 5. 언어 명령 비교 실험
- 동일한 이미지에 **다른 언어 명령**을 주면 action이 어떻게 달라지는지 비교
- 이걸 통해 SmolVLA가 언어를 단순히 무시하는 게 아니라 **실제로 이해하고 행동을 바꾼다**는 걸 확인할 수 있음
- 만약 명령이 달라도 action이 똑같다면 → 언어를 무시하는 모델
- 명령마다 action이 다르게 나온다면 → 언어를 이해하는 진짜 VLA

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 비교할 언어 명령 3가지 (같은 이미지에 다른 명령)
tasks_to_compare = [
    "Insert the peg into the socket.",
    "Move the arm to the left.",
    "Open the gripper.",
]

all_actions = {}  # 명령별 action 저장

for task in tasks_to_compare:
    policy.reset()
    task_actions = []

    tokens = processor.tokenizer(
        task,
        return_tensors="pt",
        padding="max_length",
        max_length=64,
        truncation=True
    )

    # 20프레임 동안 프레임마다 다른 이미지 + 다른 명령으로 추론
    for i in range(20):
        sample = dataset[i]        # ← 매번 다른 프레임
        image_input = sample["observation.images.top"].unsqueeze(0).to(device)
        state_input = sample["observation.state"].unsqueeze(0).to(device)

        batch = {
            "observation.images.camera1": image_input,
            "observation.images.camera2": image_input,
            "observation.images.camera3": image_input,
            "observation.state": state_input,
            "task": [task],
            "observation.language.tokens": tokens["input_ids"].to(device),
            "observation.language.attention_mask": tokens["attention_mask"].bool().to(device),
        }

        with torch.no_grad():
            action = policy.select_action(batch)

        task_actions.append(action.squeeze(0).cpu().numpy())

    all_actions[task] = np.array(task_actions)  # (20, 6)
    print(f"✅ 완료: '{task}'")

print("\nAll command inferences complete")

## Step 5 결과 시각화
- x축: 시간(프레임), y축: action 값
- 관절 6개 중 **Joint 1~3** (위치 관련)만 비교 → 차이가 가장 잘 보임
- 명령마다 선 색이 다름
- 선이 **서로 다른 패턴**을 보이면 → SmolVLA가 언어를 이해하고 있다는 증거

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle(
    "SmolVLA: Action Comparison by Language Command\n"
    "(Same image, different language instructions)",
    fontsize=12
)

joint_names = ["Joint 1 (X)", "Joint 2 (Y)", "Joint 3 (Z)",
               "Joint 4 (Roll)", "Joint 5 (Pitch)", "Joint 6 (Yaw)"]
colors = ["tab:blue", "tab:orange", "tab:green"]
# 명령 이름 줄여서 범례에 표시
short_labels = ["Insert peg", "Move left", "Open gripper"]

for j, ax in enumerate(axes.flat):
    for i, (task, actions) in enumerate(all_actions.items()):
        ax.plot(actions[:, j], color=colors[i],
                label=short_labels[i], linewidth=2, marker='o', markersize=3)
    ax.set_title(joint_names[j])
    ax.set_xlabel("Frame (time →)")
    ax.set_ylabel("Action Value")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("smolvla_command_comparison.png", dpi=100)
plt.show()
print("✅ smolvla_command_comparison.png successfully generated.")